# PhoBERT with the article body - 5-fold cross-validation (Kaggle)

**Before running**

1. Accelerator **GPU**, Internet **On**. Code and processed data are cloned from GitHub (push your latest changes first).
2. **Attach a dataset containing `bodies.jsonl`** (the file written by `python -m utils.collect_bodies`, in `data/bodies/`). Upload it as a **private** Kaggle dataset: it holds the publishers' article text and is deliberately not in the GitHub repo.

The model input becomes title + lead + the start of the article body (max length 256, PhoBERT's limit). Same 5 folds as every earlier run. Articles without a usable body keep title + lead only.

Reference: text only (title + lead), out-of-fold F1 0.771.

In [ ]:
!pip install -q pyvi

In [ ]:
import sys
from pathlib import Path

REPO_URL = "https://github.com/Zaidve/TMG_project_re.git"
CLONE_DIR = Path("/tmp/TMG_project_re")  # outside /kaggle/working so it is not saved as output
ON_KAGGLE = Path("/kaggle/input").exists()
SEARCH_ROOTS = [Path("/kaggle/input"), CLONE_DIR]

def find(pattern):
    return [hit for root in SEARCH_ROOTS if root.exists() for hit in sorted(root.rglob(pattern))]

def find_parent(pattern, levels):
    """Directory `levels` above the first matching file in an attached dataset or the clone."""
    hits = find(pattern)
    if not hits:
        raise FileNotFoundError(f"{pattern} not found under {[str(r) for r in SEARCH_ROOTS]}")
    return hits[0].parents[levels]

if ON_KAGGLE and not find("trainer/train_phobert_cv.py"):
    !git clone --depth 1 {REPO_URL} {CLONE_DIR}

if ON_KAGGLE:
    CODE_DIR = find_parent("trainer/train_phobert_cv.py", 1)
    DATA_DIR = find_parent("train.jsonl", 0)
    OUT_DIR = Path("/kaggle/working/body_cv")
else:  # local run from the notebooks/ folder
    CODE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    DATA_DIR = CODE_DIR / "data" / "processed"
    OUT_DIR = CODE_DIR / "outputs" / "body_cv"

sys.path.insert(0, str(CODE_DIR))
print("code:", CODE_DIR)
print("data:", DATA_DIR)
print("out: ", OUT_DIR)

In [ ]:
import torch

print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

## Locate the bodies file

In [ ]:
import json
from collections import Counter

roots = [Path("/kaggle/input"), CODE_DIR / "data" / "bodies"]
hits = [p for r in roots if r.exists() for p in sorted(r.rglob("bodies.jsonl"))]
assert hits, "bodies.jsonl not found - attach the dataset that contains it"
BODY_PATH = hits[0]
status = Counter(json.loads(line)["status"] for line in open(BODY_PATH, encoding="utf-8") if line.strip())
print(BODY_PATH, f"{BODY_PATH.stat().st_size / 1e6:.1f} MB")
print(dict(status))

## Train

About 15-20 minutes on a T4 (inputs are twice as long as before).

In [ ]:
from trainer.train_phobert_cv import CVConfig, run

BASE = dict(
    data_dir=str(DATA_DIR),
    n_folds=5,
    use_lead=True,
    batch_size=32,
    lr=2e-5,
    layer_decay=0.9,
    epochs=5,
    weighted_loss=True,
    patience=3,
    monitor="loss",
    seed=42,
    keep_checkpoints=False,
    limit=0,              # set to e.g. 64 for a quick smoke test
)
RUNS = {
    "text_with_body": dict(body_path=str(BODY_PATH), body_words=250, max_length=256),
}

results = {}
for name, overrides in RUNS.items():
    print(f"################ {name} ################")
    results[name] = run(CVConfig(**BASE, **overrides, out_dir=str(OUT_DIR / name)))

## Compare with text only

The fair comparison is on the articles that **have a body**: both models are scored on exactly those articles. `all articles` includes the ones where the body model saw only title + lead.

In [ ]:
import numpy as np
import pandas as pd
from trainer.trainer import classification_metrics

TEXT_DIR = CODE_DIR / "outputs" / "text_baseline"      # PhoBERT on title + lead, same folds
text = {"oof": pd.read_csv(TEXT_DIR / "oof_predictions.csv"), "test": pd.read_csv(TEXT_DIR / "test_predictions.csv")}

def scores(table, col="prob_clickbait"):
    m = classification_metrics((table["label"] == "clickbait").to_numpy(), (table[col] >= 0.5).to_numpy())
    return {"n": len(table), "f1": m["f1"], "precision": m["precision"], "recall": m["recall"], "accuracy": m["accuracy"]}

SHOW = next(iter(results))
result = results[SHOW]
pairs = {}
for split, key in (("oof", "oof_predictions"), ("test", "test_predictions")):
    pairs[split] = result[key][["id", "label", "source", "has_body", "prob_clickbait"]].merge(
        text[split][["id", "prob_clickbait"]], on="id", suffixes=("_body", "_text"), validate="one_to_one")

rows = {}
for split, pair in pairs.items():
    for subset, part in (("all articles", pair), ("with body", pair[pair["has_body"]]), ("without body", pair[~pair["has_body"]])):
        rows[(split, subset, "text only")] = scores(part, "prob_clickbait_text")
        rows[(split, subset, "text + body")] = scores(part, "prob_clickbait_body")
comparison = pd.DataFrame(rows).T
display(comparison.round(4))
print("share of articles with a body by source (out-of-fold):")
display(pairs["oof"].groupby("source")["has_body"].agg(["mean", "size"]).round(3).T)

## Is the gain real?

Paired bootstrap on the out-of-fold articles that have a body. If the 95% interval of the F1 difference includes 0, the body does not measurably help.

In [ ]:
def f1(y, pred):
    tp = (y & pred).sum()
    return 2 * tp / max(2 * tp + (~y & pred).sum() + (y & ~pred).sum(), 1)

rng = np.random.default_rng(0)
for label, pair in (("out-of-fold, with body", pairs["oof"][pairs["oof"]["has_body"]]), ("out-of-fold, all", pairs["oof"]),
                    ("test, with body", pairs["test"][pairs["test"]["has_body"]])):
    y = (pair["label"] == "clickbait").to_numpy()
    t, b = (pair["prob_clickbait_text"] >= 0.5).to_numpy(), (pair["prob_clickbait_body"] >= 0.5).to_numpy()
    idx = rng.integers(0, len(y), (2000, len(y)))
    diffs = np.array([f1(y[i], b[i]) - f1(y[i], t[i]) for i in idx])
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    print(f"{label} ({len(y)} articles): body {f1(y, b):.4f} vs text {f1(y, t):.4f} | difference {f1(y, b) - f1(y, t):+.4f} "
          f"(95% interval {lo:+.4f} to {hi:+.4f}) | fixed {((t != y) & (b == y)).sum()}, broken {((t == y) & (b != y)).sum()}")

## Per-fold results

`val_*` is each model on its own held-out fold; `test_*` is the same model alone on the test split.

In [ ]:
folds = result["fold_metrics"]
display(folds.round(4))
folds.agg(["mean", "std"]).round(4)

## Cross-validated score vs single model vs ensemble

In [ ]:
summary = pd.DataFrame(result["summary"]).T.loc[["oof", "single_model_test_mean", "single_model_test_std", "ensemble_test"]]
summary.index = ["out-of-fold (pooled train+val)", "single model on test (mean)", "single model on test (std)", "ensemble on test"]
summary.round(4)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.bar([f"fold {i}" for i in folds.index], folds["test_f1"], color="#9ecae1", label="single model")
ax.axhline(result["summary"]["ensemble_test"]["f1"], color="#E45756", linestyle="--", label="ensemble")
ax.set_ylim(max(0, folds["test_f1"].min() - 0.1), 1)
ax.set_ylabel("test F1 (clickbait)")
ax.legend(frameon=False)
fig.tight_layout()

## Ensemble on test

In [ ]:
test = result["test_predictions"]
pd.crosstab(test["label"], test["pred"], rownames=["actual"], colnames=["predicted"], margins=True)

### Threshold

Chosen on the out-of-fold predictions (never on test), then applied to the ensemble.

In [ ]:
oof = result["oof_predictions"]
oof_y, test_y = (oof["label"] == "clickbait").to_numpy(), (test["label"] == "clickbait").to_numpy()

rows = []
for t in np.arange(0.2, 0.81, 0.05):
    o = classification_metrics(oof_y, (oof["prob_clickbait"] >= t).to_numpy())
    e = classification_metrics(test_y, (test["prob_clickbait"] >= t).to_numpy())
    rows.append({"threshold": round(t, 2), "oof_precision": o["precision"], "oof_recall": o["recall"], "oof_f1": o["f1"],
                 "test_precision": e["precision"], "test_recall": e["recall"], "test_f1": e["f1"]})
thresholds = pd.DataFrame(rows).set_index("threshold")
print("best threshold on out-of-fold F1:", thresholds["oof_f1"].idxmax())
thresholds.round(3)

### Out-of-fold results per source and category

Computed on all pooled articles, so the groups are large enough to compare.

In [ ]:
def by_group(table, col):
    rows = {}
    for name, g in table.groupby(col):
        m = classification_metrics((g["label"] == "clickbait").to_numpy(), (g["pred"] == "clickbait").to_numpy())
        rows[name] = {"n": len(g), "clickbait_rate": (g["label"] == "clickbait").mean(), **m}
    return pd.DataFrame(rows).T.round(3).sort_values("n", ascending=False)

by_group(oof, "source")

In [ ]:
by_group(oof, "category")

### Most confident out-of-fold mistakes

Each prediction comes from a model that never saw the article, so confident disagreements are good candidates for a label review.

In [ ]:
pd.set_option("display.max_colwidth", 140)
wrong = oof[oof["label"] != oof["pred"]].copy()
wrong["confidence"] = (wrong["prob_clickbait"] - 0.5).abs() + 0.5
print(len(wrong), "out-of-fold mistakes;", (wrong["confidence"] >= 0.9).sum(), "with confidence >= 0.9")
wrong.sort_values("confidence", ascending=False).head(30)[["id", "source", "title", "label", "pred", "prob_clickbait"]]

## Saved files

`summary.json`, `fold_metrics.csv`, `oof_predictions.csv`, `test_predictions.csv`, `tokenizer/`, per-fold `history.json`. On Kaggle they are under `/kaggle/working/body_cv/<run name>` and appear in the Output tab after **Save Version**.

In [ ]:
for p in sorted(OUT_DIR.rglob("*")):
    if p.is_file():
        print(f"{p.stat().st_size / 1e6:8.1f} MB  {p.relative_to(OUT_DIR)}")